# Challenge 04 - Speaker Recognition (GMM)

Recognition of the speaker (31 people) of each audio track using only the MFCCs. A universal GMM (UBM) is trained on the frames of all speakers and adapted to each speaker (MAP adaptation of the means and weights). The test tracks are assigned to the speakers using the mean log-likelihoods of the GMMs, with the constraint that every speaker receives the same number of tracks. The predictions are saved in `Es6Predictions.txt`.

## Setup

In [1]:
import os
os.environ['LOKY_MAX_CPU_COUNT'] = '4'

import copy
import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.signal import savgol_filter
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

# Configuration
db_path = '../../DBs/AudioDigits_ML26_27'
train_file = 'audio_ml2627_train.pkl'
test_file = 'audio_ml2627_test_no_labels.pkl'
result_path = '../Es6Predictions.txt'

delta_width = 21
n_components = 128
relevance = 16
reg_covar = 1e-3
val_seeds = (1, 2)
seeds = [1, 2, 3, 4, 5]

VARIANTS = [('topq', 0.20), ('topq', 0.25), ('topq', 0.30), ('topq', 0.35)]

## Helper Functions

Dataset loading, preprocessing of the MFCCs, training of the UBM and of the adapted speaker GMMs, prediction and export.

Preprocessing of each track: deltas and delta-deltas are computed (21-frame window), only the frames with the highest c0 are kept (top quantile, the speech) and then c0 is dropped. The features are standardized with a scaler fitted on the training frames.

In [2]:
def load_dataset(db_path, filename):
    return np.load(os.path.join(db_path, filename), allow_pickle=True)


def delta(m, order):
    w = delta_width
    if len(m) < w:
        w = len(m) if len(m) % 2 == 1 else len(m) - 1
    if w <= order + 1:
        return np.zeros_like(m)
    return savgol_filter(m, w, polyorder=order, deriv=order, axis=0, mode='interp')

def select_frames(c0, mode, p):
    if mode == 'topq':
        return c0 >= np.quantile(c0, 1 - p)
    return c0 > c0.max() - p

def preprocess(mfccs, mode, p):
    tracks = []
    for m in mfccs:
        m = np.asarray(m, dtype=float)
        x = np.hstack([m, delta(m, 1), delta(m, 2)])
        keep = select_frames(m[:, 0], mode, p)
        if keep.sum() >= 20:
            x = x[keep]
        tracks.append(x[:, 1:])
    return tracks


def standardize(train_tracks, test_tracks):
    scaler = StandardScaler().fit(np.vstack(train_tracks))
    return [scaler.transform(m) for m in train_tracks], [scaler.transform(m) for m in test_tracks]


def map_adapt(ubm, frames):
    gmm = copy.deepcopy(ubm)
    resp = ubm.predict_proba(frames)
    n = resp.sum(axis=0) + 1e-10
    alpha = (n / (n + relevance))[:, None]
    gmm.means_ = alpha * (resp.T @ frames / n[:, None]) + (1 - alpha) * ubm.means_
    weights = alpha[:, 0] * n / n.sum() + (1 - alpha[:, 0]) * ubm.weights_
    gmm.weights_ = weights / weights.sum()
    return gmm


def speaker_scores(tr_tracks, tr_spk, te_tracks, seed):
    ubm = GaussianMixture(n_components=n_components, covariance_type='diag', reg_covar=reg_covar,
                          max_iter=200, random_state=seed).fit(np.vstack(tr_tracks))
    spk = np.unique(tr_spk)
    models = [map_adapt(ubm, np.vstack([m for m, p in zip(tr_tracks, tr_spk) if p == s])) for s in spk]
    return spk, np.array([[g.score(m) for g in models] for m in te_tracks])


def variant_scores(raw_tr, y_tr, raw_te, mode, seeds):
    x_tr, x_te = standardize(preprocess(raw_tr, *mode), preprocess(raw_te, *mode))
    total = 0
    for s in seeds:
        spk, sc = speaker_scores(x_tr, y_tr, x_te, s)
        total = total + sc
    return spk, total


def fuse(per_variant):
    norm = lambda sc: (sc - sc.mean(1, keepdims=True)) / sc.std(1, keepdims=True)
    return sum(norm(v) for v in per_variant.values())


def balanced_assign(scores, spk):
    per_spk = len(scores) // len(spk)
    assert per_spk * len(spk) == len(scores)
    rows, cols = linear_sum_assignment(np.repeat(scores, per_spk, axis=1), maximize=True)
    pred = np.empty(len(scores), dtype=int)
    pred[rows] = spk[cols // per_spk]
    return pred


def save_predictions(predictions, path):
    np.savetxt(path, np.asarray(predictions, dtype=int), fmt='%i')
    print(f'Predictions successfully saved to {path}')

## Data Loading

The training set contains sessions 1 and 4 (620 tracks, 20 per speaker), the test set sessions 2 and 3 (620 tracks, without labels). Only the MFCCs are used.

In [3]:
train_data = load_dataset(db_path, train_file)
test_data = load_dataset(db_path, test_file)

train_x = list(train_data['mfccs'])
train_y = np.array(train_data['person_ids'])
train_sess = np.array(train_data['session_ids'])
test_x = list(test_data['mfccs'])

print('Training tracks: %d, test tracks: %d, speakers: %d' % (len(train_x), len(test_x), len(np.unique(train_y))))
print('MFCCs per frame: %d, frames per track: %d-%d' % (train_x[0].shape[1], min(len(m) for m in train_x), max(len(m) for m in train_x)))

Training tracks: 620, test tracks: 620, speakers: 31
MFCCs per frame: 20, frames per track: 79-745


## Model: GMM-UBM with MAP Adaptation

A `GaussianMixture` with 128 diagonal components is trained on the frames of all speakers (UBM). The GMM of each speaker is the UBM with the means and the weights moved towards the speaker's frames, weighted by how many frames each component receives (relevance factor 16). The scores are averaged over 5 UBMs trained with different seeds.

Four systems are trained, keeping the top 20%, 25%, 30% and 35% of the frames of each track. For each track the 31 speaker scores of every system are z-normalized and summed.

## Model Evaluation

The split is done by session to simulate new recording conditions: train on one session and validate on the other, in both directions. The accuracy is shown both with the plain argmax and with the balanced assignment (10 tracks per speaker in each validation session).

In [4]:
for a, c in [(1, 4), (4, 1)]:
    tr, va = train_sess == a, train_sess == c
    raw_tr = [m for m, t in zip(train_x, tr) if t]
    raw_va = [m for m, v in zip(train_x, va) if v]
    per = {}
    for mode in VARIANTS:
        spk, per[mode] = variant_scores(raw_tr, train_y[tr], raw_va, mode, val_seeds)
    scores = fuse(per)
    acc = np.mean(spk[scores.argmax(1)] == train_y[va])
    acc_bal = np.mean(balanced_assign(scores, spk) == train_y[va])
    print('Train session %d -> validation session %d: accuracy %.4f (argmax), %.4f (balanced)' % (a, c, acc, acc_bal))

per = {}
for mode in VARIANTS:
    spk, per[mode] = variant_scores(train_x, train_y, test_x, mode, seeds)

Train session 1 -> validation session 4: accuracy 0.8097 (argmax), 0.8871 (balanced)
Train session 4 -> validation session 1: accuracy 0.8355 (argmax), 0.8581 (balanced)


## Prediction and Export

The models are retrained on all training tracks (sessions 1 and 4). The text of the exercise states that the test set contains 2 x 31 x 10 tracks (sessions 2 and 3), so every speaker has 20 test tracks. Instead of taking the argmax of each track separately, the tracks are assigned to the speakers so that each speaker gets exactly 20 tracks and the total fused score is maximal (assignment problem solved with `scipy.optimize.linear_sum_assignment`). The predictions are saved in `Es6Predictions.txt`, one speaker id per line.

In [5]:
scores = fuse(per)
predictions = balanced_assign(scores, spk)
print('Tracks changed by the balanced assignment: %d' % np.sum(predictions != spk[scores.argmax(1)]))
assert len(predictions) == 620
save_predictions(predictions, result_path)

Tracks changed by the balanced assignment: 97
Predictions successfully saved to ../Es6Predictions.txt
